Environment Setup

In [1]:
import sys
import os

if 'google.colab' in sys.modules:
    print("Detected Google Colab. Bypassing dependency resolver...")

    !pip install timm
    !pip install git+https://github.com/IBM/terratorch.git
else:
    print("Running locally. Assuming repository is already downloaded.")

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(os.path.abspath(os.getcwd()))

Detected Google Colab. Bypassing dependency resolver...
  Cloning https://github.com/IBM/terratorch.git to /tmp/pip-req-build-zg7azeb5
  Running command git clone --filter=blob:none --quiet https://github.com/IBM/terratorch.git /tmp/pip-req-build-zg7azeb5
  Resolved https://github.com/IBM/terratorch.git to commit d1267fbd278cd23bc05a1c22cc077e18481dd7c0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

Note: Please restart the Colab session before proceeding to avoid dependency conflicts.

In [1]:
print("\nCloning the official repository...")
!git clone https://github.com/karol140903/terramind-adversarial-robustness.git

%cd terramind-adversarial-robustness


Cloning the official repository...
Cloning into 'terramind-adversarial-robustness'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 159 (delta 56), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 30.21 MiB | 11.21 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/terramind-adversarial-robustness


Global Setup & Imports

In [2]:
import torch
import pandas as pd
from tqdm.notebook import tqdm
from terratorch import BACKBONE_REGISTRY

# Import core modules
from src.attacks import pgd_cos
from src.metrics import compute_metrics
from src.utils import set_seed
from src.dataset import BANDS_ORDER

# Lock the global environment
set_seed(42)
print("Global environment locked with fixed seed (42).")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing computations on: {device}")

Global environment locked with fixed seed (42).
Executing computations on: cuda


Scaling Laws Experiment Loop

In [3]:
# Define the models to evaluate scaling laws
MODELS_TO_EVALUATE = [
    "terramind_v1_tiny",
    "terramind_v1_small",
    "terramind_v1_base",
    "terramind_v1_large"
]

# Define target patches representing different semantic terrains
PATCH_PATHS = [
    {"id": "04_city", "path": "data/patches/patch_04_city.pt"},
    {"id": "09_water", "path": "data/patches/patch_09_water.pt"},
    {"id": "18_forest", "path": "data/patches/patch_18_forest.pt"}
]

# Attack hyperparameters
EPSILON = 0.01
PGD_ALPHA = 0.002
PGD_STEPS = 10

experiment_results = []

for model_name in MODELS_TO_EVALUATE:
    print(f"\n--- Evaluating {model_name.upper()} ---")

    # Reset seed to ensure deterministic parameter initialization for each model
    set_seed(42)

    # Load model dynamically
    model = BACKBONE_REGISTRY.build(model_name, pretrained=True, modalities=["S2L2A"])
    model.eval()
    model = model.to(device)

    for patch_info in tqdm(PATCH_PATHS, desc=f"Attacking {model_name}"):
        patch_id = patch_info["id"]
        tensor_path = patch_info["path"]

        # Load the predefined 12-channel tensor
        x_orig = torch.load(tensor_path, weights_only=True).to(device)

        # Lock seed immediately before the attack for perfect noise reproducibility
        set_seed(42)

        # Execute PGD Attack
        x_adv = pgd_cos(model, x_orig, epsilon=EPSILON, alpha=PGD_ALPHA, steps=PGD_STEPS)

        # Evaluate representation degradation
        cos_sim, l2_dist = compute_metrics(model, x_orig, x_adv)

        experiment_results.append({
            "Model_Size": model_name,
            "Patch_ID": patch_id,
            "Attack_Type": "PGD_Cos",
            "Cosine_Similarity": cos_sim,
            "L2_Distance": l2_dist
        })

    # Free VRAM before loading the next model to prevent CUDA Out Of Memory errors
    del model
    torch.cuda.empty_cache()

# Export results for analysis
import os
os.makedirs("results", exist_ok=True)
csv_path = "results/02_Scaling_Laws_Results.csv"

df_results = pd.DataFrame(experiment_results)
df_results.to_csv(csv_path, index=False)
print(f"\nExperiment completed successfully. Results saved to {csv_path}")

display(df_results)


--- Evaluating TERRAMIND_V1_TINY ---


TerraMind_v1_tiny.pt: reconstructing file:   0%|          |  0.00B /  212MB            

TerraMind_v1_tiny.pt: downloading bytes:           |  0.00B            

Attacking terramind_v1_tiny:   0%|          | 0/3 [00:00<?, ?it/s]


--- Evaluating TERRAMIND_V1_SMALL ---


...

TerraMind_v1_large.pt: reconstructing file:   0%|          |  0.00B / 3.79GB            

TerraMind_v1_large.pt: downloading bytes:           |  0.00B            

Attacking terramind_v1_large:   0%|          | 0/3 [00:00<?, ?it/s]


Experiment completed successfully. Results saved to results/02_Scaling_Laws_Results.csv


,Model_Size,Patch_ID,Attack_Type,Cosine_Similarity,L2_Distance
0,terramind_v1_tiny,04_city,PGD_Cos,0.587031,141.538452
1,terramind_v1_tiny,09_water,PGD_Cos,0.694896,105.239914
2,terramind_v1_tiny,18_forest,PGD_Cos,0.594271,138.360321
3,terramind_v1_small,04_city,PGD_Cos,0.486488,266.428772
4,terramind_v1_small,09_water,PGD_Cos,0.213690,330.176453
5,terramind_v1_small,18_forest,PGD_Cos,0.482223,266.527771
6,terramind_v1_base,04_city,PGD_Cos,0.235284,583.816650
7,terramind_v1_base,09_water,PGD_Cos,0.113857,667.530701
8,terramind_v1_base,18_forest,PGD_Cos,0.251409,570.529358
9,terramind_v1_large,04_city,PGD_Cos,0.614906,314.368744
